In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
import pickle
from PIL import Image

In [ ]:
def unpickle(file):
    with open(file, 'rb') as fo:
        dict = pickle.load(fo, encoding='bytes')
    return dict

data_batch_1 = unpickle("./data/CIFAR-10/data_batch_1")
data_batch_2 = unpickle("./data/CIFAR-10/data_batch_2")
data_batch_3 = unpickle("./data/CIFAR-10/data_batch_3")
data_batch_4 = unpickle("./data/CIFAR-10/data_batch_4")
data_batch_5 = unpickle("./data/CIFAR-10/data_batch_5")
test_batch = unpickle("./data/CIFAR-10/test_batch")
x_train = np.concatenate((data_batch_1[b'data'], data_batch_2[b'data'], data_batch_3[b'data'], data_batch_4[b'data'], data_batch_5[b'data']), axis=0)
y_train = np.concatenate((data_batch_1[b'labels'], data_batch_2[b'labels'], data_batch_3[b'labels'], data_batch_4[b'labels'], data_batch_5[b'labels']), axis=0)
x_test = test_batch[b'data']
y_test = np.array(test_batch[b'labels'])

In [ ]:
def rgb_to_bw(img):
    weights = np.array([0.299, 0.587, 0.114])
    gray = np.dot(img[..., :3], weights)/255.0
    return gray.astype(np.float32)

In [ ]:
print("x_train shape before conversion:", x_train.shape)
print("x_test shape before conversion:", x_test.shape)

x_train_gray = np.array([rgb_to_bw(x_train[i].reshape(3, 32, 32).transpose(1, 2, 0)) for i in range(x_train.shape[0])])
mean = x_train_gray.mean()  # scalaire unique (plus de canaux RGB)
std = x_train_gray.std()
x_train_gray = ((x_train_gray - mean) / std).reshape(x_train_gray.shape[0], -1)

x_test_gray = np.array([rgb_to_bw(x_test[i].reshape(3, 32, 32).transpose(1, 2, 0)) for i in range(x_test.shape[0])])
mean = x_test_gray.mean()  # scalaire unique (plus de canaux RGB)
std = x_test_gray.std()
x_test_gray = ((x_test_gray - mean) / std).reshape(x_test_gray.shape[0], -1)

print("x_train shape after conversion:", x_train_gray.shape)
print("x_test shape after conversion:", x_test_gray.shape)

<h1 style="text-align:center;"><b>Neural Network</b></h1>

In [ ]:
class MLP(nn.Module):
    def __init__(self, H=3):
        super().__init__()
        self.input_layer = nn.Linear(1024, 512)
        layers = []
        layers.append(nn.Linear(512, 256))
        layers.append(nn.Linear(256, 128))
        self.hidden_layers = nn.ModuleList(layers)
        self.output_layer = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.input_layer(x))
        for layer in self.hidden_layers:
            x = F.relu(layer(x))
        return self.output_layer(x)

In [ ]:
model = MLP()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4)
x_train_tensor = torch.tensor(x_train_gray, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
x_test_tensor = torch.tensor(x_test_gray, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [ ]:
def train(model, criterion, optimizer, x_train, y_train, x_test, y_test, epochs=10, batch_size=256):
    for epoch in range(epochs):
        for i in tqdm(range(0, len(x_train), batch_size)):
            x_batch = x_train[i : i + batch_size]
            y_batch = y_train[i : i + batch_size]
            model.train()
            optimizer.zero_grad()
            output = model(x_batch)
            loss = criterion(output, y_batch)
            loss.backward()
            optimizer.step()

            model.eval()
            with torch.no_grad():
                test_output = model(x_test)
                test_loss = criterion(test_output, y_test)
                pred = test_output.argmax(dim=1)
                accuracy = (pred == y_test).float().mean().item()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}, Test Loss: {test_loss.item():.4f}, Accuracy: {accuracy:.4f}")

In [ ]:
train(model, criterion, optimizer, x_train_tensor, y_train_tensor, x_test_tensor, y_test_tensor)

In [ ]:
model.eval()